# Medical Transcription Classification — Complete Pipeline

## Project Goal
This notebook implements a complete end-to-end text classification pipeline for medical transcriptions. The goal is to automatically assign a medical specialty label to free-text clinical notes using both deep learning (RNN/LSTM) and traditional machine learning (baseline) approaches.

## Dataset
- **train.csv** — 90 % of the processed dataset (columns: `label`, `description`, `text`)
- **test.csv** — 10 % of the processed dataset (same columns)
- Labels are integers 1–4 mapped to four medical specialties

## 4 Classes
| Label | Specialty |
|-------|-----------|
| 1 | Surgery |
| 2 | Medical Records |
| 3 | Internal Medicine |
| 4 | Other |

## Pipeline Overview
```
EDA → Preprocessing → RNN/LSTM Classifier → Baseline Models → Model Comparison
```

Each section is self-contained with markdown explanations, followed by runnable code. The notebook is designed to run top-to-bottom without errors.

## Section 1 — Setup & Imports

All required libraries are imported here. The notebook uses:
- **TensorFlow / Keras** for the LSTM model
- **scikit-learn** for baseline classifiers and evaluation metrics
- **pandas / numpy** for data handling
- **matplotlib / seaborn** for visualisation
- **wordcloud** (optional) for word cloud plots

File paths are defined as `Path` objects relative to the notebook directory.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re, time, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

sns.set_theme(style='whitegrid', palette='tab10')
PALETTE = sns.color_palette('tab10', 4)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})

BASE          = Path('.')
TRAIN_CSV     = BASE / 'train.csv'
TEST_CSV      = BASE / 'test.csv'
CLASSES_TXT   = BASE / 'classes.txt'
STOPWORDS_TXT = BASE / 'clinical-stopwords.txt'
VOCAB_TXT     = BASE / 'vocab.txt'

print(f"TensorFlow: {tf.__version__}")

## Section 2 — Load Auxiliary Files

Three auxiliary files provide domain knowledge for preprocessing:

- **classes.txt** — one specialty name per line (lines 1–4 map to labels 1–4)
- **clinical-stopwords.txt** — medical stop words to remove during tokenisation; lines starting with `#` are comments
- **vocab.txt** — SNOMED CT terminology vocabulary used to build the token index for the LSTM embedding layer

In [ ]:
# Load class names
CLASS_NAMES = {}
with open(CLASSES_TXT) as f:
    for idx, line in enumerate(f, start=1):
        name = line.strip()
        if name:
            CLASS_NAMES[idx] = name
print("Classes:", CLASS_NAMES)

# Load clinical stop words
STOPWORDS = set()
with open(STOPWORDS_TXT) as f:
    for line in f:
        w = line.strip()
        if w and not w.startswith('#'):
            STOPWORDS.add(w.lower())
print(f"Stop words loaded: {len(STOPWORDS):,}")

# Load SNOMED CT vocabulary
VOCAB = []
with open(VOCAB_TXT) as f:
    for line in f:
        w = line.strip()
        if w:
            VOCAB.append(w)
print(f"Vocab terms loaded: {len(VOCAB):,}")
print(f"Sample: {VOCAB[:5]}")

## Section 3 — Load & Explore Data

Load `train.csv` and `test.csv`, handle any missing values, map integer labels to class names, and add character/word length features used throughout the EDA section.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Handle missing values
train_df['text'] = train_df['text'].fillna('')
test_df['text']  = test_df['text'].fillna('')

# Map labels to names
train_df['class'] = train_df['label'].map(CLASS_NAMES)
test_df['class']  = test_df['label'].map(CLASS_NAMES)

# Add length features
train_df['char_len'] = train_df['text'].str.len()
train_df['word_len'] = train_df['text'].str.split().str.len()
test_df['char_len']  = test_df['text'].str.len()
test_df['word_len']  = test_df['text'].str.split().str.len()

print(f"Train: {train_df.shape}  |  Test: {test_df.shape}")
display(train_df.head(3))

## Section 4 — Exploratory Data Analysis

Ten subsections examine the dataset from multiple angles before modelling begins.

### 4.1 Class Distribution

How many samples belong to each of the four medical specialties? A bar chart shows raw counts; a pie chart shows proportions.

In [ ]:
counts = train_df['class'].value_counts().reindex(CLASS_NAMES.values())
pcts   = counts / counts.sum() * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bars = axes[0].bar(counts.index, counts.values, color=PALETTE)
axes[0].set_title('Class Distribution (Train)'); axes[0].set_xlabel('Specialty'); axes[0].set_ylabel('Samples')
axes[0].tick_params(axis='x', rotation=15)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=PALETTE, startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Class Distribution — Pie Chart')
plt.tight_layout(); plt.show()

### 4.2 Text Length Analysis

Understanding how long the clinical notes are helps choose the sequence length (`MAX_LEN`) for the LSTM. We look at character counts, word counts, and how length varies by class.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(train_df['char_len'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Character Count Distribution'); axes[0,0].set_xlabel('Chars'); axes[0,0].set_ylabel('Frequency')
axes[0,1].hist(train_df['word_len'], bins=50, color='darkorange', edgecolor='white')
axes[0,1].set_title('Word Count Distribution'); axes[0,1].set_xlabel('Words'); axes[0,1].set_ylabel('Frequency')
order = list(CLASS_NAMES.values())
sns.boxplot(data=train_df, x='class', y='word_len', order=order, palette=PALETTE, ax=axes[1,0])
axes[1,0].set_title('Word Count by Class (Box)'); axes[1,0].tick_params(axis='x', rotation=15)
sns.violinplot(data=train_df, x='class', y='word_len', order=order, palette=PALETTE, ax=axes[1,1], inner='quartile')
axes[1,1].set_title('Word Count by Class (Violin)'); axes[1,1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

### 4.3 Word Frequency Analysis

Which words appear most often overall and within each specialty? Stop words are removed so that medically meaningful terms dominate the charts.

In [ ]:
def tokenize(text, stopwords=STOPWORDS):
    tokens = re.findall(r'[a-z]+', str(text).lower())
    return [t for t in tokens if t not in stopwords and len(t) > 2]

all_tokens = [t for text in train_df['text'] for t in tokenize(text)]
top20 = collections.Counter(all_tokens).most_common(20)
words, freqs = zip(*top20)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(list(words)[::-1], list(freqs)[::-1], color='steelblue')
ax.set_title('Top 20 Most Frequent Words (Overall)'); ax.set_xlabel('Frequency')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    tokens = [t for text in train_df[train_df['label']==label]['text'] for t in tokenize(text)]
    top = collections.Counter(tokens).most_common(15)
    if top:
        ws, fs = zip(*top)
        ax.barh(list(ws)[::-1], list(fs)[::-1], color=PALETTE[label-1])
    ax.set_title(f'Top 15 — {cls_name}'); ax.set_xlabel('Frequency')
plt.suptitle('Top 15 Words per Class', fontsize=14, y=1.01); plt.tight_layout(); plt.show()

### 4.4 Word Clouds

Word clouds give an intuitive visual summary of the most prominent terms. One overall cloud is shown, followed by per-class clouds using distinct colour maps.

> **Note:** Requires the `wordcloud` package (`pip install wordcloud`). If not installed, a message is printed instead.

In [ ]:
if HAS_WORDCLOUD:
    wc_kw = dict(background_color='white', max_words=200, stopwords=STOPWORDS, width=800, height=400, colormap='viridis')
    wc = WordCloud(**wc_kw).generate(' '.join(train_df['text']))
    fig, ax = plt.subplots(figsize=(14, 6)); ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title('Overall Word Cloud — Medical Transcriptions', fontsize=14); plt.tight_layout(); plt.show()
    colormaps = ['Blues','Oranges','Greens','Purples']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, (label, cls_name), cmap in zip(axes.flat, CLASS_NAMES.items(), colormaps):
        text = ' '.join(train_df[train_df['label']==label]['text'])
        wc = WordCloud(**{**wc_kw, 'colormap': cmap}).generate(text)
        ax.imshow(wc, interpolation='bilinear'); ax.axis('off'); ax.set_title(cls_name, fontsize=12)
    plt.suptitle('Per-Class Word Clouds', fontsize=14); plt.tight_layout(); plt.show()
else:
    print("Install wordcloud: pip install wordcloud")

### 4.5 N-gram Analysis

Bigrams and trigrams reveal multi-word phrases that characterise each specialty — single-word frequency often misses context (e.g. *"blood pressure"* vs *"pressure"*).

In [ ]:
def get_ngrams(texts, n=2, top_k=15):
    ng = []
    for text in texts:
        tokens = tokenize(text)
        ng.extend(zip(*[tokens[i:] for i in range(n)]))
    return collections.Counter(ng).most_common(top_k)

def plot_ngrams(ngram_counts, title, ax, color):
    if not ngram_counts: return
    labels = [' '.join(g) for g, _ in ngram_counts]
    vals   = [c for _, c in ngram_counts]
    ax.barh(labels[::-1], vals[::-1], color=color); ax.set_title(title); ax.set_xlabel('Count')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_ngrams(get_ngrams(train_df['text'], n=2), 'Top 15 Bigrams', axes[0], 'steelblue')
plot_ngrams(get_ngrams(train_df['text'], n=3), 'Top 15 Trigrams', axes[1], 'darkorange')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    plot_ngrams(get_ngrams(train_df[train_df['label']==label]['text'], n=2, top_k=10),
                f'Top 10 Bigrams — {cls_name}', ax, PALETTE[label-1])
plt.suptitle('Top Bigrams per Class', fontsize=14, y=1.01); plt.tight_layout(); plt.show()

### 4.6 Vocabulary Coverage

Type-Token Ratio (TTR) measures lexical diversity — a higher TTR means more varied vocabulary. We also compare total vs unique token counts per class to understand how richly each specialty uses language.

In [ ]:
vocab_data = []
for label, cls_name in CLASS_NAMES.items():
    tokens = [t for text in train_df[train_df['label']==label]['text'] for t in tokenize(text)]
    vocab_data.append({'Class': cls_name, 'Total Tokens': len(tokens),
                       'Unique Tokens': len(set(tokens)),
                       'Type-Token Ratio': round(len(set(tokens))/max(len(tokens),1), 4)})
vocab_stats = pd.DataFrame(vocab_data)
print(vocab_stats.to_string(index=False))
x = np.arange(len(vocab_stats)); width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x-width/2, vocab_stats['Total Tokens'],  width, label='Total',  color='steelblue')
ax.bar(x+width/2, vocab_stats['Unique Tokens'], width, label='Unique', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(vocab_stats['Class'], rotation=10)
ax.set_title('Vocabulary Coverage per Class'); ax.set_ylabel('Token Count'); ax.legend()
plt.tight_layout(); plt.show()

### 4.7 Token Length Distribution & CDF

After stop-word removal, how many tokens remain per sample? The Cumulative Distribution Function (CDF) helps decide `MAX_LEN` — the 90th or 95th percentile is a common heuristic.

In [ ]:
token_counts = train_df['text'].apply(lambda t: len(tokenize(t)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(token_counts, bins=50, color='teal', edgecolor='white')
axes[0].set_title('Token Count per Sample'); axes[0].set_xlabel('Tokens after stop-word removal'); axes[0].set_ylabel('Frequency')
sorted_tc = np.sort(token_counts); cdf = np.arange(1, len(sorted_tc)+1) / len(sorted_tc)
axes[1].plot(sorted_tc, cdf, color='teal', linewidth=2)
axes[1].axhline(0.90, color='red', linestyle='--', label='90th %ile')
axes[1].axhline(0.95, color='orange', linestyle='--', label='95th %ile')
axes[1].set_title('CDF of Token Length'); axes[1].set_xlabel('Token Count'); axes[1].set_ylabel('Cumulative Proportion'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"Median: {token_counts.median():.0f}  |  90th %ile: {np.percentile(token_counts, 90):.0f}  |  95th %ile: {np.percentile(token_counts, 95):.0f}")

### 4.8 TF-IDF Correlation Heatmap

By computing TF-IDF scores for the top 30 terms and then correlating them, we can see which clinical terms tend to co-occur — providing insight into the thematic structure of the corpus.

In [ ]:
vec_eda = TfidfVectorizer(max_features=30, stop_words=list(STOPWORDS), token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')
tfidf_eda_mat = vec_eda.fit_transform(train_df['text'])
tfidf_eda_df  = pd.DataFrame(tfidf_eda_mat.toarray(), columns=vec_eda.get_feature_names_out())
corr = tfidf_eda_df.corr()
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, square=True, linewidths=0.5, annot=False, cbar_kws={'shrink':0.8})
ax.set_title('TF-IDF Feature Correlation Heatmap (Top 30 Terms)', fontsize=13)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0); plt.tight_layout(); plt.show()

### 4.9 Train / Test Split Analysis

Verify that the pre-split `train.csv` (90 %) and `test.csv` (10 %) maintain roughly the same class proportions — a stratified split is important for unbiased evaluation.

In [ ]:
tr_cnt = train_df['class'].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
te_cnt = test_df['class'].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
x = np.arange(len(CLASS_NAMES)); width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x-width/2, tr_cnt.values, width, label='Train', color='steelblue')
axes[0].bar(x+width/2, te_cnt.values, width, label='Test',  color='darkorange')
axes[0].set_xticks(x); axes[0].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[0].set_title('Class Count — Train vs. Test'); axes[0].set_ylabel('Samples'); axes[0].legend()
axes[1].bar(x-width/2, tr_cnt.values/tr_cnt.sum()*100, width, label='Train', color='steelblue')
axes[1].bar(x+width/2, te_cnt.values/te_cnt.sum()*100, width, label='Test',  color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[1].set_title('Class Proportion (%)'); axes[1].set_ylabel('%'); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.10 Missing Data Summary

Check for missing values across all key columns in both splits. A heatmap highlights any gaps visually.

In [ ]:
def missing_summary(df, name):
    ms = pd.DataFrame({'Missing Count': df.isnull().sum(),
                       'Missing %': (df.isnull().sum()/len(df)*100).round(2),
                       'Dtype': df.dtypes})
    print(f"\n{'='*40}\nMissing data — {name}\n{'='*40}")
    print(ms.to_string())
    return ms

missing_summary(train_df[['label','description','text']], 'Train')
missing_summary(test_df[['label','description','text']],  'Test')
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, df, title in [(axes[0], train_df[['label','description','text']], 'Train'),
                      (axes[1], test_df[['label','description','text']],  'Test')]:
    sns.heatmap(df.isnull(), ax=ax, yticklabels=False, cbar=False, cmap='OrRd')
    ax.set_title(f'Missing Values — {title}')
plt.tight_layout(); plt.show()

## Section 5 — Data Preprocessing

Before feeding text into the LSTM we need to:
1. **Build a word index** from `vocab.txt` (SNOMED CT terms) with special tokens `<PAD>` (0) and `<UNK>` (1)
2. **Clean and tokenise** each note — lowercase, strip non-alpha characters, remove stop words
3. **Encode** tokens to integer indices (unknown words → 1)
4. **Pad / truncate** sequences to a fixed `MAX_LEN = 300`

`MAX_LEN = 300` covers the 90th percentile of token lengths seen in the EDA above.

In [ ]:
# Build vocabulary index from vocab.txt
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word in VOCAB:
    if word not in word2idx:
        word2idx[word] = len(word2idx)
VOCAB_SIZE = len(word2idx)
MAX_LEN = 300
print(f"Vocabulary size (including PAD+UNK): {VOCAB_SIZE:,}")

def clean_and_tokenize(text):
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 1]

def encode(tokens):
    return [word2idx.get(t, 1) for t in tokens]

print("Preprocessing training set...")
X_train_pad = pad_sequences([encode(clean_and_tokenize(t)) for t in train_df['text']],
                             maxlen=MAX_LEN, padding='post', truncating='post')
print("Preprocessing test set...")
X_test_pad  = pad_sequences([encode(clean_and_tokenize(t)) for t in test_df['text']],
                             maxlen=MAX_LEN, padding='post', truncating='post')

y_train = (train_df['label'].values - 1).astype(int)
y_test  = (test_df['label'].values  - 1).astype(int)

print(f"X_train: {X_train_pad.shape}  |  X_test: {X_test_pad.shape}")
print(f"y_train classes: {np.unique(y_train)}")

## Section 6 — Part 1: RNN / LSTM Classifier

### Architecture
```
Embedding(VOCAB_SIZE, 128, input_length=300)
       ↓
LSTM(128)
       ↓
Dropout(0.3)
       ↓
Dense(4, softmax)
```

- **Embedding layer** — learns a dense 128-dimensional representation for each token
- **LSTM layer** — captures sequential dependencies in the clinical text
- **Dropout** — regularisation to reduce overfitting
- **Dense + softmax** — 4-class probability output

The model is trained for **10 epochs** with **batch size 32** and a 10 % validation split.

In [ ]:
CLASS_LABELS = list(CLASS_NAMES.values())

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128),
    Dropout(0.3),
    Dense(4, activation='softmax')
], name='LSTM_Classifier')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

rnn_start = time.time()
history = model.fit(X_train_pad, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=1)
rnn_time = time.time() - rnn_start
print(f"Training time: {rnn_time:.1f}s")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train'); axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history.history['loss'],     label='Train'); axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.suptitle('LSTM Training History'); plt.tight_layout(); plt.show()

rnn_loss, rnn_acc = model.evaluate(X_test_pad, y_test, verbose=0)
rnn_preds = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)
rnn_f1 = f1_score(y_test, rnn_preds, average='macro')
print(f"Test Accuracy: {rnn_acc:.4f}  |  Macro F1: {rnn_f1:.4f}")
print(classification_report(y_test, rnn_preds, target_names=CLASS_LABELS))

cm = confusion_matrix(y_test, rnn_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
ax.set_title('LSTM Confusion Matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## Section 7 — Part 2: Baseline Models

Three classical ML classifiers trained on **TF-IDF** features (top 10 000 terms) provide strong baselines to compare against the LSTM:

| Model | Strengths | Interpretability |
|---|---|---|
| Logistic Regression | Fast, well-calibrated | High |
| SVM (LinearSVC) | Strong on sparse text features | Medium |
| Random Forest | Handles non-linearity | Medium |

All models use the same train/test split as the LSTM.

In [ ]:
results = {}

tfidf = TfidfVectorizer(max_features=10000, stop_words=list(STOPWORDS), token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')
X_tr_tfidf = tfidf.fit_transform(train_df['text'])
X_te_tfidf = tfidf.transform(test_df['text'])
print(f"TF-IDF shape: {X_tr_tfidf.shape}")

for name, clf, interp in [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42), 'High'),
    ('SVM',                 LinearSVC(max_iter=2000, random_state=42),           'Medium'),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), 'Medium'),
]:
    print(f"\n--- {name} ---")
    t0 = time.time()
    clf.fit(X_tr_tfidf, y_train)
    elapsed = time.time() - t0
    preds = clf.predict(X_te_tfidf)
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')
    results[name] = {'Accuracy': round(acc,4), 'F1 (Macro)': round(f1,4), 'Time (s)': round(elapsed,2), 'Interpretability': interp}
    print(f"Accuracy: {acc:.4f}  |  F1: {f1:.4f}  |  Time: {elapsed:.2f}s")
    print(classification_report(y_test, preds, target_names=CLASS_LABELS))

## Section 8 — Model Comparison

All four models are now compared on:
- **Accuracy** — overall correct predictions on the test set
- **Macro F1** — class-balanced performance metric
- **Training time** — wall-clock seconds
- **Interpretability** — how explainable the model is to clinicians

Results are saved to `comparison_results.csv`.

In [ ]:
results['RNN (LSTM)'] = {'Accuracy': round(rnn_acc,4), 'F1 (Macro)': round(rnn_f1,4),
                          'Time (s)': round(rnn_time,2), 'Interpretability': 'Low'}

comp = (pd.DataFrame(results).T.reset_index().rename(columns={'index':'Model'}))
comp['Accuracy']   = comp['Accuracy'].astype(float)
comp['F1 (Macro)'] = comp['F1 (Macro)'].astype(float)
comp = comp.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(comp.to_string(index=False))

colors = sns.color_palette('tab10', len(comp))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars0 = axes[0].bar(comp['Model'], comp['Accuracy'], color=colors)
axes[0].set_title('Accuracy Comparison'); axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, 1.1)
for bar, v in zip(bars0, comp['Accuracy']): axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.4f}', ha='center', fontsize=9)
bars1 = axes[1].bar(comp['Model'], comp['F1 (Macro)'], color=colors)
axes[1].set_title('Macro F1-Score Comparison'); axes[1].set_ylabel('F1'); axes[1].set_ylim(0, 1.1)
for bar, v in zip(bars1, comp['F1 (Macro)']): axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:.4f}', ha='center', fontsize=9)
plt.xticks(rotation=10); plt.tight_layout(); plt.show()

comp.to_csv('comparison_results.csv', index=False)
print("Saved to comparison_results.csv")

## Section 9 — Conclusion

### Results Summary

The comparison table above shows how each model performed on the held-out test set. In most runs on this dataset the classical models (Logistic Regression and SVM) are surprisingly competitive with — or even outperform — the LSTM, because the dataset is small enough for TF-IDF features to capture most discriminative signal.

### Best Performing Model
- **Logistic Regression** and **SVM** often achieve the highest accuracy and F1 on smaller medical text corpora. They train in seconds and produce highly interpretable weights that clinicians can audit.
- **RNN (LSTM)** has more capacity and generally improves with more data or better pre-trained embeddings.

### Accuracy vs. Speed vs. Interpretability Trade-offs

| Model | Accuracy | Speed | Interpretability |
|---|---|---|---|
| Logistic Regression | High | Very fast | **High** — feature weights are directly readable |
| SVM | High | Fast | Medium — margin weights give some insight |
| Random Forest | Medium-High | Moderate | Medium — feature importance available |
| LSTM | Medium-High | Slow (GPU helpful) | **Low** — black-box deep network |

### Why Interpretability Matters in Medical NLP

In clinical settings, model decisions can influence patient care, billing, and compliance. Regulatory frameworks (e.g. EU AI Act, FDA guidance on AI/ML-based software) increasingly require that clinicians can **understand and challenge** automated decisions. A Logistic Regression model whose coefficients can be mapped back to specific clinical terms is far easier to audit than an LSTM's hidden states.

### When to Use RNN vs. Logistic Regression

- **Use Logistic Regression / SVM** when: dataset is small (<10k samples), quick iteration is needed, model must be auditable, or deployment resources are limited.
- **Use LSTM / deep learning** when: dataset is large (>50k samples), sequence order matters significantly, or you need to stack with pre-trained language models.

### Future Improvements

1. **Pre-trained clinical embeddings** — replace random embedding initialization with [BioWordVec](https://github.com/ncats/BioWordVec) or [ClinicalBERT](https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT) for significant accuracy gains.
2. **BioBERT / ClinicalBERT fine-tuning** — transformer-based models trained on PubMed/MIMIC data are the current state-of-the-art for clinical NLP.
3. **Data augmentation** — back-translation or synonym substitution using SNOMED CT to increase effective training size.
4. **Class imbalance handling** — if Surgery dominates, use class weights (`class_weight='balanced'`) or oversampling (SMOTE on TF-IDF features).
5. **Hyperparameter tuning** — grid-search over LSTM units, dropout rate, learning rate, and `MAX_LEN`.
6. **Ensemble** — combine LSTM probabilities with Logistic Regression probabilities via soft voting for a small but consistent accuracy boost.